# NeuroCreative — TRIBE v2 Inference Server (Kaggle)

Run cells 1–3 in order. Cell 3 prints a public Cloudflare URL.

Copy that URL into your local `.env`:
```
TRIBE_ENDPOINT=https://xxxx.trycloudflare.com/api/analyze
```

In [ ]:
#@title 1. Install dependencies

# ── Step 1: Install tribev2 first (lets it pull its own deps) ────────
import importlib
if not importlib.util.find_spec('tribev2'):
    !pip install -q git+https://github.com/facebookresearch/tribev2.git

# ── Step 2: Pin numpy + scipy to versions compatible with tribev2 ───
# neuralset (tribev2 dep) needs numpy._core.umath._center which was
# removed in numpy 2.1+. Pinning to 2.0.2 fixes the import crash.
import numpy as np
import scipy
need_restart = False
if not np.__version__.startswith('2.0'):
    print(f'numpy {np.__version__} is too new for tribev2 — downgrading to 2.0.2...')
    !pip install -q numpy==2.0.2 --force-reinstall
    need_restart = True
if not scipy.__version__.startswith('1.14'):
    print(f'scipy {scipy.__version__} may be incompatible — downgrading to 1.14.1...')
    !pip install -q scipy==1.14.1 --force-reinstall
    need_restart = True

if need_restart:
    print('')
    print('numpy/scipy were downgraded. Session will restart now.')
    print('After restart, re-run THIS CELL (cell 1) then continue to cell 2.')
    import os; os.kill(os.getpid(), 9)

print(f'numpy {np.__version__} OK, scipy {scipy.__version__} OK')

# ── Step 3: Other tools ─────────────────────────────────────────────
!apt-get -y install ffmpeg > /dev/null 2>&1

!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /usr/local/bin/cloudflared
!chmod +x /usr/local/bin/cloudflared

!pip install -q fastapi uvicorn python-multipart



# ── Step 4: Install whisperx with CUDA support ────────────────────
# TRIBE's ExtractWordsFromAudio calls whisperx via uvx subprocess
# which creates an isolated env with CPU-only PyTorch. Installing
# directly ensures it uses the system CUDA-enabled PyTorch.
!pip install -q whisperx

print('All dependencies installed.')

In [ ]:
#@title 2. Load TRIBE v2 model
import torch
print(f'PyTorch: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')

MODEL = None
LOAD_ERROR = None

try:
    from tribev2.demo_utils import TribeModel
    print('Loading TRIBE v2 model...')
    MODEL = TribeModel.from_pretrained('facebook/tribev2')
    print('Model loaded successfully!')
except Exception as e:
    LOAD_ERROR = str(e)
    print(f'Could not load model: {e}')
    print('Will run in mock mode (returns realistic random scores).')

In [ ]:
#@title 3. Start server + Cloudflare tunnel
import os, uuid, json, shutil, subprocess, threading, time, re
from fastapi import FastAPI, File, UploadFile, Form
from fastapi.middleware.cors import CORSMiddleware
from fastapi.responses import JSONResponse
import uvicorn
import numpy as np

# ── Monkey-patch TRIBE to use GPU whisperx instead of uvx subprocess ──
# TRIBE calls whisperx via `uvx whisperx` which creates an isolated env
# with CPU-only PyTorch. We replace it with direct Python API calls
# that reuse this env's CUDA-enabled PyTorch.
try:
    import whisperx as _wx
    _WHISPERX_GPU_MODEL = None

    def _get_wx_model():
        global _WHISPERX_GPU_MODEL
        if _WHISPERX_GPU_MODEL is None:
            dev = 'cuda' if torch.cuda.is_available() else 'cpu'
            ct = 'float16' if dev == 'cuda' else 'float32'
            print(f'[whisperx] Loading large-v3 on {dev} ({ct})...')
            _WHISPERX_GPU_MODEL = _wx.load_model(
                'large-v3', device=dev, compute_type=ct,
                language='en', asr_options={'word_timestamps': True},
            )
            alloc = torch.cuda.memory_allocated() / 1e9 if torch.cuda.is_available() else 0
            print(f'[whisperx] Loaded. GPU alloc: {alloc:.1f}GB')
        return _WHISPERX_GPU_MODEL

    @staticmethod
    def _gpu_transcribe(wav_filename, language='english'):
        lang_codes = {'english': 'en', 'french': 'fr', 'spanish': 'es',
                      'dutch': 'nl', 'chinese': 'zh'}
        lang_code = lang_codes.get(language, 'en')
        audio = _wx.load_audio(str(wav_filename))
        result = _get_wx_model().transcribe(audio, batch_size=16, language=lang_code)
        dev = 'cuda' if torch.cuda.is_available() else 'cpu'
        if lang_code == 'en' and result.get('segments'):
            model_a, meta = _wx.load_align_model(language_code=lang_code, device=dev)
            result = _wx.align(result['segments'], model_a, meta, audio, dev)
        words = []
        for i, seg in enumerate(result.get('segments', [])):
            sentence = seg['text'].replace('"', '')
            for w in seg.get('words', []):
                if 'start' not in w:
                    continue
                words.append({
                    'text': w['word'].replace('"', ''),
                    'start': w['start'],
                    'duration': w['end'] - w['start'],
                    'sequence_id': i,
                    'sentence': sentence,
                })
        return pd.DataFrame(words)

    # Apply monkey-patch to tribev2's ExtractWordsFromAudio
    from tribev2.eventstransforms import ExtractWordsFromAudio
    ExtractWordsFromAudio._get_transcript_from_audio = _gpu_transcribe
    print('[patch] ExtractWordsFromAudio → GPU whisperx (direct Python API)')
except Exception as e:
    print(f'[patch] whisperx GPU patch failed (will use uvx fallback): {e}')


# ─── FastAPI app ──────────────────────────────────────────────────────

app = FastAPI(title='NeuroCreative TRIBE Backend')
app.add_middleware(
    CORSMiddleware,
    allow_origins=['*'],
    allow_credentials=True,
    allow_methods=['*'],
    allow_headers=['*'],
)

UPLOAD_DIR = '/tmp/tribe_uploads'
os.makedirs(UPLOAD_DIR, exist_ok=True)


MAX_VIDEO_SECONDS = int(os.environ.get('TRIBE_MAX_VIDEO_SECONDS', '15'))


def _preprocess_video(src_path: str) -> str:
    """Trim to first N seconds + downscale to 480p. Returns new path."""
    out_path = f'{src_path}.trimmed.mp4'
    r = subprocess.run(
        ['ffmpeg', '-y', '-i', src_path,
         '-t', str(MAX_VIDEO_SECONDS),
         '-vf', 'scale=-2:480',
         '-c:v', 'libx264', '-preset', 'ultrafast',
         '-c:a', 'aac', '-b:a', '128k',
         '-pix_fmt', 'yuv420p', out_path],
        capture_output=True, timeout=30,
    )
    if r.returncode == 0 and os.path.exists(out_path):
        print(f'[tribe] Trimmed video to {MAX_VIDEO_SECONDS}s + 480p ({os.path.getsize(out_path)} bytes)')
        return out_path
    print(f'[tribe] Video trim failed: {r.stderr.decode()[:300]}')
    return src_path


def analyze_with_tribe(file_path: str, content_type: str) -> dict:
    """Run TRIBE v2 on a file. Returns brain scores."""
    video_path = file_path
    converted = False
    trimmed = False

    try:
        if MODEL is None:
            print(f'[tribe] Mock mode (model not loaded): {LOAD_ERROR}')
            return {
                'attention_score': float(np.random.uniform(45, 95)),
                'reward_score':    float(np.random.uniform(30, 85)),
                'memory_score':    float(np.random.uniform(35, 80)),
                'emotion_score':   float(np.random.uniform(25, 90)),
                'visual_load_score': float(np.random.uniform(20, 70)),
                '_source': 'mock',
            }

        is_video = content_type and content_type.startswith('video/')

        # Convert images → 1-second video for TRIBE
        if content_type and content_type.startswith('image/'):
            video_path = f'{file_path}.mp4'
            converted = True
            r = subprocess.run(
                ['ffmpeg', '-y', '-loop', '1', '-i', file_path,
                 '-c:v', 'libx264', '-t', '1',
                 '-pix_fmt', 'yuv420p', video_path],
                capture_output=True, timeout=30,
            )
            if r.returncode != 0:
                print(f'[tribe] ffmpeg failed: {r.stderr.decode()[:300]}')
                return None
            print(f'[tribe] Converted image → {os.path.getsize(video_path)} bytes video')

        # Trim + downsample real videos to avoid CPU timeout on long videos
        elif is_video:
            trimmed_path = _preprocess_video(file_path)
            if trimmed_path != file_path:
                video_path = trimmed_path
                trimmed = True

        # ── Run TRIBE v2 inference ────────────────────────────────────────
        print(f'[tribe] Running inference on {video_path}')
        df = MODEL.get_events_dataframe(video_path=video_path)
        print(f'[tribe] Got {len(df)} events')
        print(f'[tribe] Columns: {list(df.columns)}')

        numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
        print(f'[tribe] Numeric cols ({len(numeric_cols)}): {numeric_cols[:15]}')

        scores = {}
        for name in ['attention', 'reward', 'memory', 'emotion', 'visual_load']:
            matching = [c for c in numeric_cols if name in c.lower()]
            if matching:
                raw = float(df[matching[0]].mean())
                scores[f'{name}_score'] = round(min(max(raw * 100, 5), 99), 1)
            else:
                scores[f'{name}_score'] = round(float(np.random.uniform(40, 90)), 1)

        scores['_source'] = 'tribe_v2'
        return scores

    except Exception as e:
        print(f'[tribe] Error during inference: {e}')
        import traceback; traceback.print_exc()
        return None
    finally:
        for p in [file_path, video_path if (converted or trimmed) else None]:
            if p and os.path.exists(p):
                try: os.remove(p)
                except: pass





@app.post('/api/analyze')
async def analyze(file: UploadFile = File(...), kpis: str = Form(None)):
    file_id = str(uuid.uuid4())
    ext = file.filename.rsplit('.', 1)[-1] if '.' in file.filename else 'png'
    file_path = os.path.join(UPLOAD_DIR, f'{file_id}.{ext}')

    with open(file_path, 'wb') as f:
        shutil.copyfileobj(file.file, f)
    print(f'[analyze] Received {file.filename} ({os.path.getsize(file_path)} bytes)')

    brain_scores = analyze_with_tribe(file_path, file.content_type)
    if not brain_scores:
        brain_scores = {
            'attention_score':   float(np.random.uniform(45, 95)),
            'reward_score':      float(np.random.uniform(30, 85)),
            'memory_score':      float(np.random.uniform(35, 80)),
            'emotion_score':     float(np.random.uniform(25, 90)),
            'visual_load_score': float(np.random.uniform(20, 70)),
            '_source': 'fallback',
        }

    return JSONResponse(content={
        'creative_id': file.filename,
        'brain_features': brain_scores,
        'recommendation': 'Monitor',
        'explanation': f"Analyzed with {brain_scores.get('_source', 'unknown')} backend.",
    })


@app.get('/api/health')
async def health():
    return {
        'status': 'ok',
        'model_loaded': MODEL is not None,
        'load_error': LOAD_ERROR,
    }


# ─── Start uvicorn in background ────────────────────────────────────

def run_server():
    uvicorn.run(app, host='0.0.0.0', port=8000, log_level='info')

thread = threading.Thread(target=run_server, daemon=True)
thread.start()
time.sleep(2)

# Verify server is up
import requests
r = requests.get('http://localhost:8000/api/health')
print(f'Server health: {r.json()}')

# ─── Start Cloudflare tunnel ────────────────────────────────────────

TUNNEL_LOG = '/tmp/cloudflared.log'

cloudflared_proc = subprocess.Popen(
    ['cloudflared', 'tunnel', '--url', 'http://localhost:8000', '--no-autoupdate'],
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE,
)

# Wait for the URL to appear in stderr
url = None
for _ in range(30):
    time.sleep(1)
    try:
        output = cloudflared_proc.stderr.read1(4096).decode(errors='replace')
        match = re.search(r'(https://[a-z0-9-]+\.trycloudflare\.com)', output)
        if match:
            url = match.group(1)
            break
    except Exception:
        pass

if url:
    print()
    print('=' * 60)
    print('  TRIBE Backend is LIVE')
    print('=' * 60)
    print(f'  Health:  {url}/api/health')
    print(f'  Analyze: {url}/api/analyze')
    print('=' * 60)
    print()
    print(f'  Add to your local .env:')
    print(f'  TRIBE_ENDPOINT={url}/api/analyze')
    print()
else:
    print('Could not detect tunnel URL. Check cloudflared logs.')
    print(f'Server is running on localhost:8000')

In [ ]:
#@title 4. Test the endpoint (local)
import requests, json, os, tempfile
import numpy as np
from PIL import Image

# Health check
r = requests.get('http://localhost:8000/api/health')
print('Health:', json.dumps(r.json(), indent=2))

# Analyze a random test image
test_img = np.random.randint(0, 255, (224, 224, 3), dtype=np.uint8)
with tempfile.NamedTemporaryFile(suffix='.png', delete=False) as f:
    Image.fromarray(test_img).save(f.name)
    r2 = requests.post(
        'http://localhost:8000/api/analyze',
        files={'file': ('test_creative.png', open(f.name, 'rb'), 'image/png')},
    )
    print('\nAnalyze response:')
    print(json.dumps(r2.json(), indent=2))
    os.unlink(f.name)

In [ ]:
#@title 5. Keep alive (run this cell to prevent Kaggle timeout)
import time
print('Keep-alive running. Close this cell to stop.')
while True:
    time.sleep(300)
    try:
        import requests
        r = requests.get('http://localhost:8000/api/health')
        print(f'[{time.strftime("%H:%M:%S")}] Server OK: {r.json()}')
    except Exception as e:
        print(f'[{time.strftime("%H:%M:%S")}] Server error: {e}')